# Add Author and Country information

In [11]:
from Bio import Entrez
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from sentence_transformers import SentenceTransformer

import datetime
import time

from sklearn.manifold import TSNE

import os
import sys

In [ ]:
year_start = int(sys.argv[1])
year_end = int(sys.argv[2])
print(f"----- Downloading PubMed data from {year_start} to {year_end} -----")

In [2]:
Entrez.email = "leetaerim1@gmail.com"  # Always tell NCBI who you are

In [4]:
def get_pubmed_data(yr, month, day, n_days_window, n_papers):

    start_date = datetime.date(yr, month, day)
    # end_date = datetime.date(yr, 2, 5)
    if n_days_window == 0:
        end_date = start_date
    else:
        end_date = start_date + datetime.timedelta(days=n_days_window)

    # print(start_date)
    # print(end_date)

    current_date = start_date

    df_out = []

    while current_date <= end_date:
        date_str = current_date.strftime("%Y/%m/%d")
        print(f"Searching for date: {date_str}...")

        query = f'("{date_str}"[Date - Create] : "{date_str}"[Date - Create]) AND (hasabstract) NOT preprint[filter]'
        print(query)

        # Perform the search
        search_handle = Entrez.esearch(
            db="pubmed",
            term=query,
            retmax=n_papers,
        )
        search_results = Entrez.read(search_handle)
        search_handle.close()

        # time.sleep(1)  # Be respectful to NCBI servers

        id_list = search_results["IdList"]
        if len(id_list) == 0:
            print(f"No results found for date: {date_str}. Moving to next date.\n")
            current_date += datetime.timedelta(days=1)
            continue
        else:
            print(f"Found {len(id_list)} results. Top 5 IDs: {id_list[:5]} ...\n")

        # Fetch details for the found IDs
        fetch_handle = Entrez.efetch(db="pubmed", id=",".join(id_list), retmode="xml")
        # print('Here 1')
        time.sleep(0.2)
        records = Entrez.read(fetch_handle)
        # print('Here 2')
        time.sleep(0.2)
        fetch_handle.close()
        # print('Here 3')
        time.sleep(0.2)

        # df_out = []
        # df_CorAuthors = []

        for article in records["PubmedArticle"]:
            df_CorAuthors = []

            # Title
            title = article["MedlineCitation"]["Article"]["ArticleTitle"]
            # print(f"Title: {title}")

            try:
                # Add affiliations
                for auth in article["MedlineCitation"]["Article"]["AuthorList"][
                    -1:
                ]:  # Only look at the last author
                    name = f"{auth.get('LastName', '')} {auth.get('ForeName', '')}"
                    affiliations = auth.get("AffiliationInfo", [])
                    for aff in affiliations[
                        :1
                    ]:  # Only look at the first affiliation for simplicity
                        aff_text = aff.get("Affiliation", "")
                        # print(f"Author: {name}, Affiliation: {aff_text}")
                        df_CorAuthors_tmp = pd.DataFrame(
                            {
                                "PMID": article["MedlineCitation"]["PMID"],
                                "Author": name,
                                "Affiliation": aff_text,
                                "Country": (
                                    aff_text.split(",")[-1].strip()
                                    if "," in aff_text
                                    else ""
                                ),
                            },
                            index=[0],
                        )
                        df_CorAuthors.append(df_CorAuthors_tmp)
                    # df_CorAuthors = pd.concat(df_CorAuthors, ignore_index=True)
                    df_CorAuthors = pd.concat(df_CorAuthors, ignore_index=True)

            except Exception as e:
                # print(
                #     f"Error processing authors/affiliations for PMID {article['MedlineCitation']['PMID']}: {e}"
                # )
                df_CorAuthors = pd.DataFrame(
                    {
                        "PMID": article["MedlineCitation"]["PMID"],
                        "Author": None,
                        "Affiliation": None,
                        "Country": None,
                    },
                    index=[0],
                )

            # Abstract
            abstract = (
                article["MedlineCitation"]["Article"]
                .get("Abstract", {})
                .get("AbstractText", [])
            )
            if abstract:
                abstract = [str(x) for x in abstract]
                if len(abstract) > 1:
                    abstract2 = "".join(abstract)
                else:
                    abstract2 = abstract[0]
            else:
                abstract2 = "No abstract available."

            df_out_tmp = pd.DataFrame(
                {
                    # "Author": author,
                    # "Affiliation": affiliation,
                    "Date": date_str,
                    "Year": date_str.split("/")[0],
                    "PMID": article["MedlineCitation"]["PMID"],
                    "title": title,
                    "abstract": abstract2,
                    "journal": article["MedlineCitation"]["Article"]["Journal"][
                        "Title"
                    ],
                    "corresponding_authors": df_CorAuthors.Author.tolist(),
                    "corresponding_affiliations": df_CorAuthors.Affiliation.tolist(),
                    "corresponding_countries": df_CorAuthors.Country.tolist(),
                },
                index=[0],
            )
            df_out.append(df_out_tmp)

        # df_CoAuthors = pd.concat(df_CoAuthors, ignore_index=True)

        # Update date for next iteration
        current_date += datetime.timedelta(days=1)
        date_str = current_date.strftime("%Y/%m/%d")
        # print(f"Updated date: {date_str}")
        # print(f"{'-'*40} Done {'-'*40}\n")

    if len(df_out) > 0:
        df_out = pd.concat(df_out, ignore_index=True)
    else:
        df_out = pd.DataFrame(
            columns=["Date", "Year", "PMID", "title", "abstract", "journal"]
        )

    return df_out

In [5]:
# Parse finished years.
DIR = "/scratch/users/taerim/projects/FindMyLab"
cmd = f"mkdir -p {DIR}/DataCollection/"
os.system(cmd)

years_finished = os.listdir(f"{DIR}/DataCollection/")
years_finished = [year for year in years_finished if year.startswith("pubmed_data_")]
years_finished = [int(year.split("_")[-1].split(".")[0]) for year in years_finished]
years_finished.sort()
print(years_finished[:5])
print(years_finished[-5:])

[1980, 1981, 1982]
[1980, 1981, 1982]


In [10]:
# day = 1
n_days_window = 0
n_papers = 10000

dict_day_by_month = {
    1: 31,
    2: 28,  # Ignoring leap years for simplicity
    3: 31,
    4: 30,
    5: 31,
    6: 30,
    7: 31,
    8: 31,
    9: 30,
    10: 31,
    11: 30,
    12: 31,
}

In [ ]:
for year in range(year_start, year_end + 1):
    if year in years_finished:
        print(f"Year {year} already processed. Skipping...")
        continue

    df_out = []
    print(f">>>>> Processing year: {year}...")

    for month in range(1, 13):
        for day in range(1, dict_day_by_month[month] + 1):
            # try up to 3 times if there is a runtime error.
            attemps = 0
            while attemps < 3:
                try:
                    df_out.append(
                        get_pubmed_data(year, month, day, n_days_window, n_papers)
                    )
                    break  # If successful, break out of the retry loop
                except Exception as e:
                    attemps += 1
                    print(f"Error occurred: {e}. Attempt {attemps}/3. Retrying...")
                    time.sleep(5)  # Wait before retrying

    # Save output for the year.
    df_out = pd.concat(df_out, axis=0, ignore_index=True)
    df_out.to_csv(f"{DIR}/DataCollection/pubmed_data_{year}.tsv", index=False, sep="\t")